# 04.5 Result Analysis

The focus of this notebook is not to train yet another model. The focus is learning how to read results. A final accuracy number is only a summary; training curves, controlled comparisons, result tables, confusion matrices, and error examples explain what kind of performance the model has.

Good result analysis helps you decide what to change next instead of guessing.

## Learning Goals

After this notebook, you should be able to:

1. improved models and compare them.
2. Use training curves to judge learning behavior.
3. Organize a small result table.
4. Use a confusion matrix to inspect class-level errors.
5. Perform simple analysis on misclassified samples.
6. Turn experimental observations into clear conclusions.

In [ ]:
import copy

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

## Prepare the Data

We use `sklearn digits` here because it is small enough for a full analysis exercise.


In [ ]:
digits = load_digits()
images = digits.images
labels = digits.target

X_train_full, X_test, y_train_full, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

train_mean = X_train.mean() / 16.0
train_std = X_train.std() / 16.0

def normalize_split(x):
    return ((x / 16.0) - train_mean) / train_std


X_train_norm = normalize_split(X_train).reshape(len(X_train), -1)
X_val_norm = normalize_split(X_val).reshape(len(X_val), -1)
X_test_norm = normalize_split(X_test).reshape(len(X_test), -1)

train_ds = TensorDataset(torch.tensor(X_train_norm, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(X_val_norm, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(X_test_norm, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

print("train size =", len(train_ds))
print("val size =", len(val_ds))
print("test size =", len(test_ds))

## Define a Baseline and an Improved Model

To keep the analysis clear, we mainly change the model complexity while keeping the rest of the workflow similar.


In [ ]:
class LinearBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        return self.fc(x)


class ImprovedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)


print(LinearBaseline())
print()
print(ImprovedMLP())

## Shared Training and Evaluation Functions

The advantage of shared training functions is that the comparison becomes fairer and easier to analyze.


In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_items = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = logits.argmax(dim=1)
            total_loss += loss.item() * xb.size(0)
            total_correct += (preds == yb).sum().item()
            total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items


def train_model(model, train_loader, val_loader, epochs=10, lr=0.01):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
            }
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    return history


def collect_predictions(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb).argmax(dim=1)
            all_preds.append(preds)
            all_targets.append(yb)
    return torch.cat(all_preds), torch.cat(all_targets)

## Run Two Experiments

We intentionally make the workflow feel like a small project. Two models are trained with the same data split and evaluated with the same criteria. That makes the comparison controlled: when results differ, the model change is the main explanation rather than a hidden change in data or evaluation.

This is the discipline behind experiment tracking.

In [ ]:
torch.manual_seed(42)
baseline_model = LinearBaseline()
baseline_history = train_model(baseline_model, train_loader, val_loader, epochs=8, lr=0.01)

print("\n--- improved model ---")
torch.manual_seed(42)
improved_model = ImprovedMLP()
improved_history = train_model(improved_model, train_loader, val_loader, epochs=10, lr=0.005)

In [ ]:
baseline_history_df = pd.DataFrame(baseline_history)
improved_history_df = pd.DataFrame(improved_history)

baseline_test_preds, baseline_test_targets = collect_predictions(baseline_model, test_loader)
improved_test_preds, improved_test_targets = collect_predictions(improved_model, test_loader)

results_df = pd.DataFrame(
    [
        {
            "model": "LinearBaseline",
            "best_val_acc": round(baseline_history_df["val_acc"].max(), 4),
            "test_acc": round(accuracy_score(baseline_test_targets, baseline_test_preds), 4),
            "epochs": len(baseline_history_df),
        },
        {
            "model": "ImprovedMLP",
            "best_val_acc": round(improved_history_df["val_acc"].max(), 4),
            "test_acc": round(accuracy_score(improved_test_targets, improved_test_preds), 4),
            "epochs": len(improved_history_df),
        },
    ]
)
results_df

## Training Curves

Training curves contain more information than a single final metric. They show whether the model is still learning, whether validation performance starts to degrade while training performance improves, and which model converges faster. This helps distinguish underfitting, overfitting, and optimization problems.

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(baseline_history_df["epoch"], baseline_history_df["train_loss"], label="baseline train")
plt.plot(baseline_history_df["epoch"], baseline_history_df["val_loss"], label="baseline val")
plt.plot(improved_history_df["epoch"], improved_history_df["train_loss"], label="improved train")
plt.plot(improved_history_df["epoch"], improved_history_df["val_loss"], label="improved val")
plt.title("Loss Curves")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(baseline_history_df["epoch"], baseline_history_df["train_acc"], label="baseline train")
plt.plot(baseline_history_df["epoch"], baseline_history_df["val_acc"], label="baseline val")
plt.plot(improved_history_df["epoch"], improved_history_df["train_acc"], label="improved train")
plt.plot(improved_history_df["epoch"], improved_history_df["val_acc"], label="improved val")
plt.title("Accuracy Curves")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.tight_layout()
plt.show()
plt.close()

## Choose the Better Model

In this notebook, we choose the final model by the higher `test_acc`.


In [ ]:
baseline_test_acc = accuracy_score(baseline_test_targets, baseline_test_preds)
improved_test_acc = accuracy_score(improved_test_targets, improved_test_preds)

if improved_test_acc >= baseline_test_acc:
    best_name = "ImprovedMLP"
    best_model = improved_model
    best_preds = improved_test_preds
    best_targets = improved_test_targets
else:
    best_name = "LinearBaseline"
    best_model = baseline_model
    best_preds = baseline_test_preds
    best_targets = baseline_test_targets

print("best model =", best_name)
print("baseline_test_acc =", round(baseline_test_acc, 4))
print("improved_test_acc =", round(improved_test_acc, 4))
print("delta =", round(improved_test_acc - baseline_test_acc, 4))

## 7. Confusion Matrix

Overall accuracy tells you how often the model is right, but not where it goes wrong.

A `confusion matrix` helps you see confusion patterns across classes.


In [ ]:
cm = confusion_matrix(best_targets, best_preds)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.title(f"Confusion Matrix: {best_name}")
plt.xlabel("predicted label")
plt.ylabel("true label")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black", fontsize=8)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
cm_no_diag = cm.copy()
for i in range(cm_no_diag.shape[0]):
    cm_no_diag[i, i] = 0

worst_pair_index = cm_no_diag.argmax()
true_digit, pred_digit = divmod(int(worst_pair_index), cm_no_diag.shape[1])
worst_pair_count = cm_no_diag[true_digit, pred_digit]

print("most confused pair / most confused class pair =", (true_digit, pred_digit))
print("count =", int(worst_pair_count))
print(classification_report(best_targets, best_preds, digits=4))

## 8. Error Analysis

Below we directly inspect several misclassified examples.


In [ ]:
best_preds_np = best_preds.numpy()
best_targets_np = best_targets.numpy()
mis_idx = [i for i, (t, p) in enumerate(zip(best_targets_np, best_preds_np)) if t != p]

print("num misclassified =", len(mis_idx))
num_show = min(6, len(mis_idx))

if num_show > 0:
    plt.figure(figsize=(10, 5))
    for plot_i, data_i in enumerate(mis_idx[:num_show], start=1):
        plt.subplot(2, 3, plot_i)
        plt.imshow(X_test[data_i], cmap="gray")
        plt.title(f"true={best_targets_np[data_i]}, pred={best_preds_np[data_i]}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close()
else:
    print("No misclassified samples / no misclassified samples.")

## How to Write an Experiment Conclusion

A practical experiment conclusion should answer at least three things:

1. which model is better
2. what evidence supports that claim
3. what you would try next

In [ ]:
summary_lines = [
    f"Best model: {best_name}",
    f"Baseline test accuracy: {baseline_test_acc:.4f}",
    f"Improved model test accuracy: {improved_test_acc:.4f}",
    f"Most confused pair: true {true_digit} -> predicted {pred_digit} (count={int(worst_pair_count)})",
]

for line in summary_lines:
    print(line)

In [ ]:
# Exercise 1
#
# Explain in full sentences:
# Why is it not enough to look only at test_acc?
#
# Your answer should mention what training curves and the confusion matrix can
# reveal that one final accuracy number hides.

Exercise 1 Reference Answer

Because a single metric tells you the overall performance, but not whether training was stable or which classes the model struggles with.


In [ ]:
# Exercise 2
#
# Suppose the improved model has very high train_acc but only a small gain in
# val_acc.
#
# What would you suspect first, and what would you try next? Answer in full
# sentences.

Exercise 2 Reference Answer

I would first suspect overfitting. The model may be memorizing the training set without learning a representation that generalizes much better. Reasonable next steps include adding regularization, reducing model complexity, using more augmentation when appropriate, or collecting more data.

## Summary

The most important takeaways from this notebook are:

1. experiment analysis should not rely on only one final number
2. training curves help you understand learning dynamics
3. a result table is useful for model comparison
4. the confusion matrix and misclassified samples help localize error patterns
5. turning observations into explicit conclusions is part of doing real project work